# P07 リモート化と認可 ― トークンで利用者を決める

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

顧客台帳のサーバを HTTP で立て、トークンの有無・署名・宛先・期限・範囲で 401 / 403 / 200 がどう変わるか、トークンの持ち主ごとに見える項目がどう変わるかを確かめます。各セルはサーバを子プロセスで立て、終わると止めます。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- API キーが無ければ、講師の実行記録を再生します (費用 0)。記録も無ければ mock で最後まで流れます (mock の数字は性能の証拠になりません)。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. Claude の呼び先を決める (キーは表示しない) と、ラボのモジュールを読めるようにする
from common.nb import choose_provider, safe
from common.kit import use_labs, utf8_console
PROVIDER = choose_provider(IN_COLAB)
use_labs(); utf8_console()

## 1. トークン無しの 401 と、保護リソースのメタデータ
401 の `WWW-Authenticate` が指す URL を開くと、どの認可サーバからどの範囲のトークンをもらえばよいかが書いてあります。

In [ ]:
from run_remote_auth import amain as remote
with safe():
    r = await remote(["--only", "noauth,prm"])

## 2. トークンを変えて当てる
署名の書き換え・宛先違い・期限切れ・範囲不足・知らない利用者・正しいトークン。🔴 署名の鍵は表示しません。トークンも先頭 12 文字だけ出します。

In [ ]:
with safe():
    r = await remote(["--only", "tokens"])

## 3. 同じ道具を、トークンの持ち主ごとに
ホストは `Host(mode='http', urls=..., tokens=...)` でつなぎます。利用者を決めるのはトークンだけです。サーバの環境変数 `SOLAHA_USER=U-04` (部長) は効きません。

In [ ]:
with safe():
    r = await remote(["--only", "users,envuser", "--users", "U-01,U-02,U-05"])

## 4. トークンの中身
講座のトークンは `v1.<中身>.<署名>` の形です。中身は鍵なしで読めます (秘密ではない)。書き換えると署名が合いません。

In [ ]:
with safe():
    r = await remote(["--only", "decode"])